# 机器修正与对比：人工审阅之后 → 机器后处理 → 四个版本对比 → 对比视频 / 对比界面

人工审阅（`review_pipeline.ipynb`）看完一个实验之后，在这里：
1. 导出最终数据：应用人工修正，再做一次**机器后处理**（去掉剩下的 1–12 帧尖峰和位置跳变），鱼的位置改用胸鳍（pectoral）节点；
2. 生成**四个版本**，用来比较每一步改了什么；
3. 检查机器修正发生在哪里（有没有把真实的快速动作当成错误）；
4. 导出对比视频；打开对比界面。

| 版本 | 内容 | 对比 |
|---|---|---|
| `initial` 初始 | 直接三角化，不做任何机器修正 | |
| `machine` 机器修正 | 审阅数据（构建时：三视角不一致时用两视角重算 + 去单帧尖峰）= 人工审阅时看到的数据 | initial → machine：机器修正前后 |
| `manual` 人工修正 | machine + 人工修正（身份 swap、头尾、各视角修正、B 重估身体、区间求解） | machine → manual：人工修正前后 |
| `final` 最终 | manual + 机器后处理 | manual → final：后处理前后；initial → final：初始 vs 全部完成 |

**为什么后处理放在人工修正之后**：剩下的位置跳变约 40% 出现在人工 swap 的那一帧（换身份时两条鱼不在同一位置），只有按“每条真实的鱼”连起来的轨迹才看得到；而且要在 B 重估身体、头尾修正之后做，才不会把人工修好的地方又改掉。

**机器后处理的规则**（`scripts/review/postprocess.py`，参数见最后一节）：
- **尖峰**：一段 ≤12 帧（≤72 ms）的“出去又回来”。用加速度 > 3 mm/帧² 找候选；用两侧正常帧的位置和速度做三次 Hermite 曲线桥接；只有原值偏离桥接线 ≥ 1.5 mm、且两侧的运动对得上（端点误差 ≤ 偏离量的一半）才替换。真实的转身、甩尾、加速能被桥接线贴住，不会被改。
- **跳变**：胸鳍单帧移动 > 6 mm（约 1 m/s，鱼做不到），在跳变两侧几帧内平滑过渡，整条鱼一起移动。
- 先修胸鳍（世界坐标），嘴和尾在“相对胸鳍”的坐标里修 —— 整条鱼游动、转身不会被当成抖动。
- 被替换的值 `<节点>_source = 5`（估计值，不是测量值）；所有替换和留下没改的 `jerk` 都写进 `postprocess_report.csv`。

**背后的脚本**：`scripts/review/export_reviewed.py`（导出）、`postprocess.py`（后处理）、`versions.py`（四个版本）、`postprocess_qc.py`（检查表）、`compare_gui.py`（对比界面 + 导出视频）。

**参数固定，不按实验调**：66 个实验用同一套参数，组间差异才只来自鱼本身，而不是处理方式。

## 步骤 1 — 参数

只需要改这一格。
- `EXPERIMENTS`：要处理的实验（完整文件夹名或 `P12`、`ALL` 这样的组名）
- `RUN_EXPORT`：`True` 时步骤 4 真正导出（每个实验约 2 分钟；第一次还要算一次 `initial`，约 1 分钟，之后用缓存）
- `FORCE_EXPORT`：`False` = 只导出“修正记录比上次导出新”的实验
- `CLIPS`：要导出的对比视频：`(实验, 起始帧, 结束帧, 版本A, 版本B, 速度)`，速度 0.25 = 慢 4 倍
- `OPEN_GUI`：`True` 时步骤 9 打开对比界面（`GUI_EXPERIMENT`）

In [1]:
EXPERIMENTS = ["20260112_P_1_1_FCMC"]
ROUND_TAG   = "s14_bottomup_3node_reviewed"
REVIEW_ROOT  = None          # None = <data_root>/review_packages
PATHS_CONFIG = None          # None = 自动选择

RUN_EXPORT   = True
FORCE_EXPORT = False
RUN_CLIPS    = True
CLIPS = [
    # (实验, 起始帧, 结束帧, A, B, 速度)
    ("20260112_P_1_1_FCMC", 87500, 87720, "manual", "final", 0.25),
]
OPEN_GUI       = False
GUI_EXPERIMENT = "20260112_P_1_1_FCMC"

## 步骤 2 — 定位仓库根目录和本机路径配置

- 输入：当前目录、`configs/paths.<用户名>.toml` 或 `configs/paths.local.toml`
- 输出：`REPO`、`PATHS`、`REVIEW_ROOT`

In [2]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.common.local_paths import load_local_paths
from scripts.sleap.batch_bottomup_track import default_paths_config

CFG = Path(PATHS_CONFIG) if PATHS_CONFIG else default_paths_config()
PATHS = load_local_paths(CFG)
if PATHS is None:
    raise FileNotFoundError(f"路径配置不存在：{CFG}")
REVIEW_ROOT = Path(REVIEW_ROOT) if REVIEW_ROOT else PATHS.data_root / "review_packages"
print("REVIEW_ROOT:", REVIEW_ROOT, "✅" if REVIEW_ROOT.exists() else "❌ 不存在")
print("仓库根目录 :", REPO)
print("路径配置   :", CFG)
print("Python     :", sys.executable)

REVIEW_ROOT: \\bucket.oist.jp\bucket\StephensU\medaka\review_packages ✅
仓库根目录 : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline
路径配置   : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\configs\paths.ReiterU.toml
Python     : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe


## 步骤 3 — 状态总表

每个实验一行（随时重跑）：
- 已看过：`coverage.csv` 里播放过或逐帧看过的比例（**任何速度都算**），从 analysis start 算起
- 修正行数：`human_events.csv` 的行数
- 导出：✅ 已导出且比修正记录新；🔄 修正记录比导出新，需要重新导出；空白 = 未导出
- 可分析：`reviewed_summary.json` 的 `ready_for_analysis`（标了起点、三条鱼都命名、全部看过、几何修正已应用）

In [3]:
import json

import pandas as pd

from scripts.common.experiment_sets import resolve_experiments
from scripts.review.ledger import COVERAGE_COLUMNS, EVENT_COLUMNS, analysis_start, covered_fraction, read_csv, watched_intervals

EXPS = resolve_experiments(EXPERIMENTS, PATHS.medaka_data_dir)


def folder_of(exp):
    return REVIEW_ROOT / exp / ROUND_TAG


def export_state(exp):
    f = folder_of(exp)
    summary, events = f / "reviewed_summary.json", f / "human_events.csv"
    if not summary.exists() or not (f / "versions.npz").exists():
        return ""
    return "🔄" if events.exists() and events.stat().st_mtime > summary.stat().st_mtime else "✅"


rows = []
for exp in EXPS:
    f = folder_of(exp)
    row = {"实验": exp, "审阅数据": "✅" if (f / "review_data.npz").exists() else ""}
    if row["审阅数据"]:
        ev = read_csv(f / "human_events.csv", EVENT_COLUMNS)
        cov = read_csv(f / "coverage.csv", COVERAGE_COLUMNS)
        import numpy as np
        with np.load(f / "review_data.npz") as z:
            last = int(z["frames"][-1])
        start = analysis_start(ev)
        row["起点"] = start
        row["已看过"] = f"{100 * covered_fraction(watched_intervals(cov), start or 0, last):.2f}%"
        row["修正行数"] = len(ev)
        row["导出"] = export_state(exp)
        s = f / "reviewed_summary.json"
        row["可分析"] = json.loads(s.read_text(encoding="utf-8")).get("ready_for_analysis") if s.exists() else ""
    rows.append(row)
pd.DataFrame(rows)

,实验,审阅数据,起点,已看过,修正行数,导出,可分析
0,20260112_P_1_1_FCMC,✅,4152,100.00%,526,✅,True


## 步骤 4 — 导出：人工修正 + 机器后处理 + 四个版本

对每个实验运行 `export_reviewed.export`，写到审阅文件夹里：
- `reviewed_3d.csv`：每帧每条鱼一行。`X/Y/Z` = 修正后的**胸鳍节点**（鱼的位置），`centroid_X/Y/Z` = 3D 重建的质心（仅供参考）；每个节点的 3D 坐标和 `<节点>_source`（0 三角化，1 单视角，2 身体模型，3 构建时去尖峰，4 人工 B 重估，5 后处理桥接，-1 缺失）
- `postprocess_report.csv`：后处理替换的每一段，和留下没改的 `jerk`（加速度大但不像尖峰：真实的快速动作，或太长修不了的错误）
- `versions.npz`：四个版本（给步骤 5–9 和对比界面用）
- `reviewed_summary.json`：可分析检查、后处理统计、节点来源比例

需要标定文件（在本机跑；笔记本上没有标定时 `initial` 版本会缺）。

In [4]:
import time

from scripts.review.export_reviewed import export

if not RUN_EXPORT:
    print("RUN_EXPORT = False：跳过导出")
for exp in EXPS if RUN_EXPORT else []:
    f = folder_of(exp)
    if not (f / "review_data.npz").exists():
        print(f"{exp}: 没有审阅数据，跳过")
        continue
    if not FORCE_EXPORT and export_state(exp) == "✅":
        print(f"{exp}: 已是最新导出，跳过（FORCE_EXPORT = True 可重做）")
        continue
    t0 = time.time()
    s = export(f, log=print)
    print(f"{exp}: 完成 {time.time() - t0:.0f} 秒；可分析 = {s['ready_for_analysis']} {s['checks']}")
    print("   后处理替换比例（分析范围内的行）:", s["postprocess"]["bridged_fraction_of_analysed_rows"])
    print("   版本:", s["versions"])

20260112_P_1_1_FCMC: 已是最新导出，跳过（FORCE_EXPORT = True 可重做）


## 步骤 5 — 四个版本的抖动对比

只看 analysis start 之后。每个版本、每个节点：
- `off_median_gt_2mm_%`：偏离 9 帧滑动中位数超过 2 mm 的 fish-frame 比例（尖峰）
- `acc_gt_3mm_%`：加速度 > 3 mm/帧² 的比例（抖动）
- `max_step_mm`：最大单帧位移（> 6 mm 基本不是真实运动）

注意：`manual` 的胸鳍最大单帧位移比 `machine` 大，是人工 swap 换身份的那一帧造成的；`final` 把这些跳变平滑掉了。

In [5]:
from scripts.review.postprocess_qc import change_breakdown, jitter_table, load_versions, measured_share

VERS = {}
for exp in EXPS:
    f = folder_of(exp)
    if (f / "versions.npz").exists():
        VERS[exp] = load_versions(f)
for exp, v in VERS.items():
    print(exp)
    display(jitter_table(v).pivot(index="node", columns="version").reindex(
        columns=["initial", "machine", "manual", "final"], level=1))

20260112_P_1_1_FCMC


off_median_gt_2mm_%                       acc_gt_3mm_%          \
version              initial machine manual  final      initial machine   
node                                                                      
mouth                  0.537   0.309  0.318  0.173        0.997   0.642   
pectoral               0.388   0.238  0.257  0.080        0.765   0.555   
tail                   1.193   0.880  0.861  0.581        2.032   1.479   

                       max_step_mm                          
version  manual  final     initial machine  manual   final  
node                                                        
mouth     0.645  0.315      25.008  25.017  25.098  21.885  
pectoral  0.581  0.167      13.464  12.115  27.105   8.162  
tail      1.450  0.791      33.712  34.871  49.564  46.085

## 步骤 6 — 机器后处理改在了哪里（误报检查）

`final` 版本里被后处理替换（source 5）的 fish-frame 比例：
- 按游速：如果高速段被改得特别多，可能有真实的快速动作被误改 —— 用对比界面的 “fast > 200 mm/s” 筛选逐个看
- 按和最近一条鱼的距离：修正集中在鱼靠得近的时候（视角里互相遮挡）
- 按 BORIS 行为：求偶 / 攻击期间的修正比例

**这对组间比较的影响**：互动多的实验，被估计（而不是测量）的帧就多。分析时做一次敏感性检验：只用测量帧（source 0）和用全部帧各算一遍，结论一致才可靠。步骤 7 给出测量帧的比例。

In [6]:
from scripts.review.behaviors import default_search, find_boris_csv, load_bouts

for exp, v in VERS.items():
    csv = find_boris_csv(exp, default_search())
    bouts = load_bouts(csv) if csv else None
    print(exp, "BORIS:", csv)
    for name, table in change_breakdown(v, bouts=bouts).items():
        print(name)
        display(table)

20260112_P_1_1_FCMC BORIS: C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\archive\boris\p2\20260112_P_1_1_FCMC.csv
by_speed_mm_s


,changed_%,fish_frames,share_of_data_%
speed_mm_s,,,
"(0.0, 25.0]",0.780,450531,43.599
"(25.0, 50.0]",0.880,301948,29.220
"(50.0, 100.0]",1.243,201781,19.527
"(100.0, 200.0]",2.504,64645,6.256
"(200.0, 300.0]",5.854,9481,0.918
"(300.0, inf]",17.282,4901,0.474


by_nearest_fish_mm


,changed_%,fish_frames,share_of_data_%
nearest_mm,,,
"(0.0, 10.0]",11.199,25262,2.445
"(10.0, 20.0]",2.549,109238,10.571
"(20.0, 40.0]",1.240,301634,29.190
"(40.0, inf]",0.393,597210,57.794


by_behaviour


,behaviour,frames,changed_%,mean_speed_mm_s
0,aggressive,37633,0.951,46.854
1,courtship,109221,1.709,46.823
2,other,609,15.380,67.466
3,(no bout),204602,0.832,39.017


## 步骤 7 — 敏感性检验的输入：三个节点都是测量值的比例

按和最近一条鱼的距离分组。距离越近，被估计的节点越多。

In [7]:
for exp, v in VERS.items():
    print(exp)
    display(measured_share(v))

20260112_P_1_1_FCMC


,all_nodes_measured_%,fish_frames
nearest_mm,,
"(0.0, 10.0]",55.973,25262
"(10.0, 20.0]",93.949,109238
"(20.0, 40.0]",97.444,301634
"(40.0, inf]",99.340,597210


## 步骤 8 — 后处理没改、值得人看一眼的地方

`postprocess_report.csv` 里（只看 analysis start 之后）：
- `jump` 且 `repaired = False`：胸鳍跳变太大（> 36 mm）或旁边太乱，没有平滑 —— 多半是身份 swap 的位置不对，用对比界面看
- 最大的 `jerk`：加速度很大但不像尖峰，可能是真实的逃逸 / 攻击，也可能是太长修不了的错误

在对比界面里用变更列表的筛选（group = post，kind = `jump pectoral (left)`）逐个跳过去看。

In [8]:
for exp in EXPS:
    rp = folder_of(exp) / "postprocess_report.csv"
    if not rp.exists():
        continue
    rep = pd.read_csv(rp)
    start = int(VERS[exp]["analysis_start"]) if exp in VERS else 0
    rep = rep[rep["frame_end"] >= start]                     # 只看分析范围（起点之前有手 / 网）
    print(exp, f"（帧 {start} 之后）：替换", int(rep["repaired"].sum()), "段，留下", int((~rep["repaired"]).sum()), "段")
    print(rep.groupby(["node", "kind", "repaired"]).size().unstack(fill_value=0))
    print("没有平滑的胸鳍跳变（按大小）：")
    display(rep[(rep["kind"] == "jump") & ~rep["repaired"]].sort_values("size_mm", ascending=False).head(15))

20260112_P_1_1_FCMC （帧 4152 之后）：替换 5949 段，留下 5850 段
repaired        False  True 
node     kind               
mouth    jerk    1145      0
         spike      0   1424
pectoral jerk    1026      0
         jump      26    238
         spike      0   1404
tail     jerk    3653      0
         spike      0   2883
没有平滑的胸鳍跳变（按大小）：


,fish_id,label,node,kind,frame_start,frame_end,size_mm,repaired
9572,ID_3,F,pectoral,jump,171020,171021,8.16,False
9614,ID_3,F,pectoral,jump,290610,290611,7.74,False
9615,ID_3,F,pectoral,jump,290611,290612,7.74,False
9616,ID_3,F,pectoral,jump,290612,290613,7.74,False
9609,ID_3,F,pectoral,jump,290599,290600,7.60,False
9608,ID_3,F,pectoral,jump,290598,290599,7.60,False
9610,ID_3,F,pectoral,jump,290600,290601,7.60,False
5001,ID_2,M1,pectoral,jump,258626,258627,7.53,False
9567,ID_3,F,pectoral,jump,171004,171005,7.39,False
9565,ID_3,F,pectoral,jump,171002,171003,7.39,False


## 步骤 9 — 导出对比视频

`CLIPS` 里的每一段写成 mp4（30 fps）：三个视角并排，A 虚线、B 实线，下面是三条鱼的速度曲线；画面自动放大到鱼身上；默认所有版本用审阅后的身份（只看几何差异）。
写到 `<审阅文件夹>/compare_clips/`。对比界面里按 E 也能导出当前看的那一段。

In [9]:
from scripts.review.compare_gui import CompareData, export_clip

if not RUN_CLIPS:
    print("RUN_CLIPS = False：跳过")
for exp, start, end, a, b, speed in CLIPS if RUN_CLIPS else []:
    t0 = time.time()
    cd = CompareData(folder_of(exp))
    out = export_clip(cd, start, end, (a, b), speed=speed)
    cd.close()
    print(f"{out}  ({time.time() - t0:.0f} 秒)")

\\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed\compare_clips\20260112_P_1_1_FCMC_manual_vs_final_87500-87720.mp4  (6 秒)


## 步骤 10 — 打开对比界面

在新窗口打开（不会挡住 notebook）。也可以自己在命令行里运行：

```powershell
.venv\Scripts\python.exe -m scripts.review.compare_gui <REVIEW_ROOT>\<实验>\<ROUND_TAG>
```

界面：
- 上方：版本 A（虚线）和 B（实线）；预设 1 机器修正前后（initial→machine）、2 人工修正前后（machine→manual）、3 后处理前后（manual→final）、4 初始 vs 全部（initial→final）
- `I` 所有版本用审阅后的身份（只看几何差异）；关掉则早期版本保持跟踪器的身份，人工 swap 也会显示成差异
- 中间：三个视角 + 可旋转的 3D；下方：曲线（速度 / |A−B| / 加速度 / 体长 / X Y Z，滚轮改时间范围，点击跳转），彩色条 = 修正（绿 构建，橙 人工，蓝 后处理）
- 右侧变更列表：按组 / 鱼 / 类型 / 快速 / 最小幅度筛选，双击或 N / P 跳转
- 键：空格 播放，← → 一帧，Shift+← → 1 秒，↑ ↓ 播放速度，Z 放大到鱼，E 导出视频，F1 帮助

In [10]:
import subprocess

if OPEN_GUI:
    folder = folder_of(GUI_EXPERIMENT)
    flags = getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0) | getattr(subprocess, "DETACHED_PROCESS", 0)
    subprocess.Popen([sys.executable, "-m", "scripts.review.compare_gui", str(folder)], cwd=REPO, creationflags=flags)
    print("已打开对比界面：", folder)
else:
    print("OPEN_GUI = False")

OPEN_GUI = False


## 参数说明（`PostParams`，所有实验相同）

| 参数 | 值 | 含义 |
|---|---|---|
| `acc_mm` | 3.0 mm/帧² | 加速度阈值（约 83 m/s²）；10 Hz、振幅 4 mm 的甩尾峰值只有 0.6 |
| `floor_mm` | 1.5 mm | 原值至少偏离桥接线这么多才替换 |
| `return_ratio` | 0.5 | 两侧运动的端点误差 ≤ 偏离量 × 0.5（“出去又回来”） |
| `max_run` | 12 帧 | 最长替换 72 ms；更长的写进报告，不改 |
| `max_step_mm` | 6.0 mm/帧 | 胸鳍单帧超过这个 = 跳变 |

改参数要对所有实验重新导出（`FORCE_EXPORT = True`），并在报告里写明。